In [ ]:
# TensorFlow
import tensorflow as tf

# Numerical operations
import numpy as np

# Visualization
import matplotlib.pyplot as plt

# File operations
import os

# Image preprocessing
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# EfficientNet model
from tensorflow.keras.applications import EfficientNetB0

# Layers and model building
from tensorflow.keras import layers, models

In [ ]:
# Define binary dataset path
dataset_path = "../dataset_binary"

# Display folders
print(os.listdir(dataset_path))

In [ ]:
# Define image size
IMG_SIZE = 224

# Batch size
BATCH_SIZE = 32

# Create image generator
datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

# Load training dataset
train_data = datagen.flow_from_directory(
    dataset_path,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='training'
)

# Load validation dataset
val_data = datagen.flow_from_directory(
    dataset_path,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='validation'
)

In [ ]:
# Load EfficientNetB0 model
base_model = EfficientNetB0(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)

# Freeze pretrained layers
base_model.trainable = False

# Build binary classifier
model = models.Sequential([

    base_model,

    layers.GlobalAveragePooling2D(),

    layers.Dense(128, activation='relu'),

    layers.Dropout(0.5),

    # Binary output layer
    layers.Dense(1, activation='sigmoid')
])

In [ ]:
# Compile binary classifier
model.compile(

    optimizer='adam',

    # Binary classification loss
    loss='binary_crossentropy',

    metrics=['accuracy']
)

In [ ]:
# Train binary classifier
history = model.fit(

    train_data,

    validation_data=val_data,

    epochs=10
)

In [ ]:
# Save binary classification model
model.save(
    "../models/binary_papsmear_model.h5"
)